# <b><span style='color:#1F2A4F'>1 | INTRODUCTION</span></b>


<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>1.1 |</span> Why class-overlapping CL?</b></p></div>

<div style="color:white;display:fill;border-radius:6px;font-size:95%;letter-spacing:0.8px;background-color:#1F2A4F;padding:6px 12px;margin:8px 0;width:fit-content"><b>PROBLEM STATEMENT</b></div>

Most continual-learning benchmarks treat tasks as disjoint — each class appears in exactly one task. Real data streams revisit classes across time.

- Standard benchmarks like Split-CIFAR-100 assign each class to a single task, creating artificially clean boundaries.
- Real deployments frequently surface the same categories at different times, with potentially different label distributions.
- Disjoint benchmarks over-estimate forgetting on well-separated tasks and under-estimate it on semantically related ones.
- Methods tuned on disjoint streams may exploit clean task boundaries that do not exist in practice.
- Class overlap is the distinguishing design axis of the CLOVER benchmark library.
- The PILOT codebase introduced `OverlapDataManager` to begin addressing this — CLOVER generalises it with a declarative API.

> <span style='color:#4A5FBF'><b>Key takeaway —</b></span> Disjoint CL benchmarks do not reflect real data streams; class revisitation is the norm, not the exception.

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>1.2 |</span> What this notebook covers</b></p></div>

- Installing and importing CLOVER; comparing its two APIs side-by-side.
- Building three benchmark streams on CIFAR-100: disjoint, partial overlap, and exact replay.
- Training a ResNet-18 replay baseline across all three scenarios in a unified loop.
- Plotting the *killer metric*: forgetting on revisiting classes vs. fresh classes.
- Discussing where simple replay falls short and what trunk-and-branch methods should improve.

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>1.3 |</span> Setup</b></p></div>

This notebook targets Kaggle's free T4 GPU (~5 min runtime) and falls back to CPU (~15—20 min). Set `N_CLASSES = 50` to reduce runtime further on slow hardware. The install cell below is a no-op if packages are already present.

In [ ]:
import subprocess, sys

def _pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

_pip("git+https://github.com/danushkumar-v/clover-cl.git")
_pip("torch")
_pip("torchvision")
_pip("matplotlib")
_pip("seaborn")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, json, random, time
from pprint import pprint

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = ["#4A5FBF", "#7B8EE8", "#1F2A4F", "#A8B5E8", "#3447A0"]
sns.set_palette(PALETTE)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlecolor"] = "#1F2A4F"
plt.rcParams["axes.labelcolor"] = "#1F2A4F"

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
N_CLASSES   = 100    # set to 50 on slow hardware
N_EPOCHS    = 5
BATCH_SIZE  = 64
BUFFER_SIZE = 200
LR          = 1e-3
SEED        = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Device: {DEVICE} | PyTorch {torch.__version__} | Classes: {N_CLASSES}")

# <b><span style='color:#1F2A4F'>2 | THE CLOVER LIBRARY</span></b>


<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>2.1 |</span> Installation</b></p></div>


In [ ]:
import clover
print(f"CLOVER {clover.__version__}")

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>2.2 |</span> Two APIs at a glance</b></p></div>

CLOVER ships two interfaces that produce equivalent benchmark objects:

- **`OverlapDataManager`** — the legacy API, compatible with PILOT's data pipeline. Familiar to users of `sun-hailong/LAMDA-PILOT`.
- **`build_benchmark(StreamSpec(...))`** — the modern declarative API. Seed-controlled, composable, and supports overlap scheduling via **`RevisitSpec`**.

Both are shown below on the same Split-CIFAR-100 disjoint configuration.

In [ ]:
# Legacy API â€” PILOT-compatible
from clover import OverlapDataManager
dm = OverlapDataManager("cifar100", init_cls=10, increment=10)
print("[Legacy API]")
print(f"  Tasks:              {dm.nb_tasks}")
print(f"  Task-0 classes:     {dm.get_task_classes(0)[:5]} ...")

In [ ]:
# Modern API â€” declarative
from clover import build_benchmark, StreamSpec
spec = StreamSpec(
    dataset="cifar100",
    init_cls=10,
    increment=10,
    revisits=[],
    shuffle_seed=SEED,
)
bench = build_benchmark(spec)
print("[Modern API]")
print(f"  Experiences:        {bench.nb_experiences}")
print(f"  Task-0 classes:     {bench.train_stream[0].classes_in_this_experience[:5]} ...")

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>2.3 |</span> First look — disjoint baseline</b></p></div>

The disjoint stream assigns 10 classes per task with no repetition — the standard Split-CIFAR-100 setup.

In [ ]:
print(f"Tasks: {bench.nb_experiences}")
for t in range(bench.nb_experiences):
    exp = bench.train_stream[t]
    cls = exp.classes_in_this_experience
    print(f"  Task {t}: {len(cls)} classes â€” {cls[:4]} ...")

# <b><span style='color:#1F2A4F'>3 | DESIGNING AN OVERLAP STREAM</span></b>


<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>3.1 |</span> StreamSpec — the declarative API</b></p></div>

`StreamSpec` captures everything needed to reproduce a benchmark stream:

- **`dataset`** — dataset name; `"cifar100"` uses torchvision under the hood.
- **`init_cls`** — number of classes in the first task.
- **`increment`** — classes added per subsequent task.
- **`revisits`** — list of `RevisitSpec` objects defining which classes re-appear and when.
- **`image_strategy`** — controls image-level overlap for the whole stream: `"disjoint"`, `"duplicate"`, or `"partial_duplicate"`.
- **`shuffle_seed`** — controls class-to-task assignment; fixes the random split for reproducibility.

<div style="color:white;display:fill;border-radius:6px;font-size:95%;letter-spacing:0.8px;background-color:#1F2A4F;padding:6px 12px;margin:8px 0;width:fit-content"><b>MODERN API</b></div>

In [ ]:
from clover import StreamSpec, RevisitSpec

# `classes=3` means "pick 3 classes at random and revisit them".
# Pass a list like [5, 12, 33] to revisit specific class indices.
spec_example = StreamSpec(
    dataset="cifar100",
    init_cls=10,
    increment=10,
    revisits=[
        RevisitSpec(classes=3, placement="random", min_gap=3, times=1),
    ],
    image_strategy="partial_duplicate",
    shuffle_seed=SEED,
)
pprint(vars(spec_example))

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>3.2 |</span> RevisitSpec — controlling class repetition</b></p></div>

`RevisitSpec` specifies when and how classes re-appear:

- **`classes`** — an integer (how many random classes to revisit) or a list of specific class indices.
- **`placement`** — `"random"` places revisits in randomly chosen tasks subject to `min_gap`; `"spaced"` distributes evenly; `"end_of_stream"` appends to the final task.
- **`min_gap`** — minimum number of tasks between original appearance and revisit.
- **`times`** — how many times to revisit.

In [ ]:
spec_revisit = StreamSpec(
    dataset="cifar100",
    init_cls=10,
    increment=10,
    revisits=[RevisitSpec(classes=1, placement="random", min_gap=3, times=1)],
    shuffle_seed=SEED,
)
bench_rv = build_benchmark(spec_revisit)
for t in range(bench_rv.nb_experiences):
    exp = bench_rv.train_stream[t]
    cls = exp.classes_in_this_experience
    revisiting = exp.revisiting_classes
    marker = f"  <-- revisit: {revisiting}" if revisiting else ""
    print(f"Task {t}: {cls[:4]} ...{marker}")

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>3.3 |</span> Image-level overlap modes</b></p></div>

Three `image_strategy` values on `StreamSpec` control what images revisiting tasks actually see:

- **`"disjoint"`** — revisiting task sees entirely fresh images (no image-level overlap).
- **`"duplicate"`** — revisiting task sees the exact same images as the original task (perfect replay).
- **`"partial_duplicate"`** — a mix: some original images, some fresh ones.

The strategy applies to the whole stream, not individual revisit specs.

In [ ]:
for strategy in ["disjoint", "duplicate", "partial_duplicate"]:
    spec_s = StreamSpec(
        dataset="cifar100",
        init_cls=10,
        increment=10,
        revisits=[RevisitSpec(classes=1, placement="random", min_gap=3, times=1)],
        image_strategy=strategy,
        shuffle_seed=SEED,
    )
    bench_s = build_benchmark(spec_s)
    print(f"\nimage_strategy='{strategy}'")
    for t in range(bench_s.nb_experiences):
        exp = bench_s.train_stream[t]
        if exp.revisiting_classes:
            print(f"  Task {t}: revisiting {exp.revisiting_classes} | samples in task: {exp.n_samples}")

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>3.4 |</span> Visualizing the resulting stream</b></p></div>


In [ ]:
bench_viz = build_benchmark(spec_example)
n_tasks   = bench_viz.nb_experiences
matrix    = np.zeros((100, n_tasks), dtype=int)
for t in range(n_tasks):
    for c in bench_viz.train_stream[t].classes_in_this_experience:
        matrix[c, t] = 1

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(matrix, ax=ax, cmap=["#FFFFFF", "#4A5FBF"],
            linewidths=0, cbar=False, xticklabels=range(n_tasks))
ax.set_xlabel("Task index")
ax.set_ylabel("Class index")
ax.set_title("Stream overlap matrix â€” partial revisit scenario")
plt.tight_layout()
plt.show()

# <b><span style='color:#1F2A4F'>4 | DATASET — SPLIT-CIFAR-100</span></b>


<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>4.1 |</span> Dataset overview</b></p></div>

- 100 classes, 32x32 RGB images.
- 500 training images per class (50,000 total); 100 test images per class (10,000 total).
- Split into 10 tasks of 10 classes each in the disjoint baseline.
- Distributed with torchvision — auto-downloaded on first run.

In [ ]:
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

full_train = torchvision.datasets.CIFAR100(
    root="./data", train=True,  download=True, transform=train_transform)
full_test  = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=test_transform)

CLASSES = full_train.classes
print(f"Train: {len(full_train)} | Test: {len(full_test)} | First 10 classes: {CLASSES[:10]}")

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>4.2 |</span> Class hierarchy and visualization</b></p></div>


In [ ]:
raw_train = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=False, transform=transforms.ToTensor())

class_to_idx = {v: [] for v in range(100)}
for idx, (_, label) in enumerate(raw_train):
    class_to_idx[label].append(idx)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i in range(16):
    ax = axes[i // 4][i % 4]
    img, _ = raw_train[class_to_idx[i][0]]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[i], fontsize=8, color="#1F2A4F")
    ax.axis("off")
plt.suptitle("CIFAR-100 sample images (classes 0--15)",
             fontsize=12, color="#1F2A4F", fontweight="bold")
plt.tight_layout()
plt.show()

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>4.3 |</span> Building three streams</b></p></div>

Three scenarios sharing the same dataset and task count:

- **Scenario A (disjoint)** — standard Split-CIFAR-100; no class revisitation.
- **Scenario B (partial overlap)** — 3 randomly chosen classes revisit once with `partial_duplicate` images.
- **Scenario C (exact replay)** — 10 randomly chosen classes re-appear at end-of-stream with `duplicate` images.

In [ ]:
# Scenario A â€” disjoint
spec_A  = StreamSpec(dataset="cifar100", init_cls=10, increment=10,
                     revisits=[], shuffle_seed=SEED)
bench_A = build_benchmark(spec_A)

# Scenario B â€” 3 random classes revisit once each, partial_duplicate images
spec_B  = StreamSpec(dataset="cifar100", init_cls=10, increment=10,
                     revisits=[RevisitSpec(classes=3, placement="random",
                                          min_gap=3, times=1)],
                     image_strategy="partial_duplicate",
                     shuffle_seed=SEED)
bench_B = build_benchmark(spec_B)

# Scenario C â€” 10 random classes re-appear at end with duplicate images
spec_C  = StreamSpec(dataset="cifar100", init_cls=10, increment=10,
                     revisits=[RevisitSpec(classes=10, placement="end_of_stream",
                                          times=1)],
                     image_strategy="duplicate",
                     shuffle_seed=SEED)
bench_C = build_benchmark(spec_C)

# Collect revisiting classes from Scenario B for §7.2
revisiting_B = set()
for t in range(bench_B.nb_experiences):
    revisiting_B.update(bench_B.train_stream[t].revisiting_classes)

print(f"A: {bench_A.nb_experiences} tasks | B: {bench_B.nb_experiences} tasks | C: {bench_C.nb_experiences} tasks")
print(f"\nScenario B revisiting classes: {sorted(revisiting_B)}")
for t in range(bench_B.nb_experiences):
    rv = bench_B.train_stream[t].revisiting_classes
    if rv:
        print(f"  Task {t} revisits: {rv}")

# <b><span style='color:#1F2A4F'>5 | THE BASELINE METHOD — REPLAY</span></b>


<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>5.1 |</span> Why a simple replay baseline</b></p></div>

Experience replay is the simplest and most widely understood CL baseline. It stores a fixed-size buffer of past examples and replays them alongside new data during training. It makes no assumptions about task boundaries and requires no architectural changes.

For this demo, replay serves as a proxy for methods that treat all tasks equally. It does not exploit class revisitation — when a class re-appears, the buffer gives it no special treatment. This is the gap that methods like GRAFT are designed to close.

> <span style='color:#4A5FBF'><b>Key takeaway —</b></span> Simple replay avoids catastrophic forgetting on buffered samples but misses the opportunity to consolidate revisiting classes.

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>5.2 |</span> Model: ResNet-18</b></p></div>


In [ ]:
def build_model(n_out=N_CLASSES):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, n_out)
    return model.to(DEVICE)

m = build_model()
n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"ResNet-18 | Parameters: {n_params:,} | Output classes: {N_CLASSES}")
del m

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>5.3 |</span> Replay buffer mechanics</b></p></div>

Reservoir sampling maintains a fixed-size buffer with uniform coverage over all past examples. Each incoming example replaces a randomly chosen slot with probability `buffer_size / n_seen`. This ensures no systematic bias toward early or late tasks.

In [ ]:
class ReplayBuffer:
    def __init__(self, max_size=BUFFER_SIZE):
        self.max_size = max_size
        self.buffer   = []
        self.n_seen   = 0

    def update(self, xs, ys):
        for x, y in zip(xs, ys):
            self.n_seen += 1
            if len(self.buffer) < self.max_size:
                self.buffer.append((x.cpu(), int(y)))
            else:
                j = random.randint(0, self.n_seen - 1)
                if j < self.max_size:
                    self.buffer[j] = (x.cpu(), int(y))

    def sample(self, n):
        if not self.buffer:
            return None, None
        chosen = random.sample(self.buffer, min(n, len(self.buffer)))
        xs = torch.stack([c[0] for c in chosen]).to(DEVICE)
        ys = torch.tensor([c[1] for c in chosen], dtype=torch.long).to(DEVICE)
        return xs, ys

    def __len__(self):
        return len(self.buffer)

buf = ReplayBuffer(max_size=5)
for i in range(12):
    buf.update([torch.zeros(3, 32, 32)], [torch.tensor(i % N_CLASSES)])
print(f"Buffer after 12 updates (max 5): size={len(buf)}, n_seen={buf.n_seen}")

# <b><span style='color:#1F2A4F'>6 | TRAINING ACROSS THREE SCENARIOS</span></b>


In [ ]:
def filter_dataset(dataset, allowed_classes):
    allowed = set(allowed_classes)
    indices = [i for i, (_, y) in enumerate(dataset) if y in allowed]
    return Subset(dataset, indices)


def evaluate(model, loaders):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for loader in loaders:
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                correct += (model(x).argmax(1) == y).sum().item()
                total   += y.size(0)
    return correct / total if total else 0.0


def evaluate_classes(model, dataset, class_ids):
    ds     = filter_dataset(dataset, class_ids)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return evaluate(model, [loader])


def train_one_scenario(benchmark, full_train, full_test, label=""):
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    buffer    = ReplayBuffer(max_size=BUFFER_SIZE)

    per_task_acc      = []
    seen_test_loaders = []
    t0 = time.time()

    for t in range(benchmark.nb_experiences):
        exp = benchmark.train_stream[t]
        cls = exp.classes_in_this_experience

        tr_ds  = filter_dataset(full_train, cls)
        te_ds  = filter_dataset(full_test,  cls)
        tr_ldr = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
        te_ldr = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        seen_test_loaders.append(te_ldr)

        model.train()
        for _ in range(N_EPOCHS):
            for x, y in tr_ldr:
                x, y = x.to(DEVICE), y.to(DEVICE)
                rx, ry = buffer.sample(BATCH_SIZE)
                if rx is not None:
                    x = torch.cat([x, rx])
                    y = torch.cat([y, ry])
                optimizer.zero_grad()
                criterion(model(x), y).backward()
                optimizer.step()

        for x, y in tr_ldr:
            buffer.update(x, y)

        acc     = evaluate(model, seen_test_loaders)
        elapsed = time.time() - t0
        print(f"  [{label}] Task {t:2d} | acc_so_far={acc:.3f} | elapsed={elapsed:.1f}s")
        per_task_acc.append(acc)

    return model, per_task_acc

print("Training helpers defined.")

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>6.1 |</span> Scenario A — disjoint</b></p></div>


In [ ]:
print("Training Scenario A (disjoint) ...")
model_A, acc_A = train_one_scenario(bench_A, full_train, full_test, label="A-disjoint")
with open("results_A.json", "w") as f:
    json.dump({"scenario": "A_disjoint", "acc_per_task": acc_A}, f)

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>6.2 |</span> Scenario B — partial overlap</b></p></div>


In [ ]:
print("Training Scenario B (partial overlap) ...")
model_B, acc_B = train_one_scenario(bench_B, full_train, full_test, label="B-partial")
with open("results_B.json", "w") as f:
    json.dump({"scenario": "B_partial", "acc_per_task": acc_B}, f)

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>6.3 |</span> Scenario C — exact replay</b></p></div>


In [ ]:
print("Training Scenario C (exact replay) ...")
model_C, acc_C = train_one_scenario(bench_C, full_train, full_test, label="C-replay")
with open("results_C.json", "w") as f:
    json.dump({"scenario": "C_exact_replay", "acc_per_task": acc_C}, f)

# <b><span style='color:#1F2A4F'>7 | RESULTS AND ANALYSIS</span></b>


<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>7.1 |</span> Per-experience accuracy curves</b></p></div>


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(len(acc_A)), acc_A, marker="o", label="A â€” disjoint",
        color="#4A5FBF", linewidth=2)
ax.plot(range(len(acc_B)), acc_B, marker="s", label="B â€” partial overlap",
        color="#7B8EE8", linewidth=2)
ax.plot(range(len(acc_C)), acc_C, marker="^", label="C â€” exact replay",
        color="#1F2A4F", linewidth=2)
ax.set_xlabel("Task index")
ax.set_ylabel("Accuracy on all seen classes")
ax.set_title("Accuracy after each task â€” three scenarios")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>7.2 |</span> Forgetting on revisited vs. fresh classes</b></p></div>

<div style="color:white;display:fill;border-radius:6px;font-size:95%;letter-spacing:0.8px;background-color:#1F2A4F;padding:6px 12px;margin:8px 0;width:fit-content"><b>KILLER METRIC</b></div>

For Scenario B we split the test set into **revisiting classes** (appeared more than once in the stream) and **fresh classes** (appeared exactly once). Final-model accuracy is reported separately for each group.

Simple replay treats all buffered examples uniformly — it does not know a class is revisiting. A method that *exploits* overlap (e.g., GRAFT-style trunk-and-branch) should show higher accuracy on revisiting classes by routing them through a consolidated specialist head.

> <span style='color:#4A5FBF'><b>Key takeaway —</b></span> The gap between revisiting-class and fresh-class accuracy is the evaluation axis that separates overlap-aware methods from overlap-agnostic ones.

In [ ]:
all_seen_B = set()
for t in range(bench_B.nb_experiences):
    all_seen_B.update(bench_B.train_stream[t].classes_in_this_experience)

fresh_B = all_seen_B - revisiting_B

acc_revisiting = evaluate_classes(model_B, full_test, list(revisiting_B))
acc_fresh      = evaluate_classes(model_B, full_test, list(fresh_B))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Revisiting classes", "Fresh classes"],
              [acc_revisiting, acc_fresh],
              color=["#4A5FBF", "#7B8EE8"], width=0.5)
for bar, val in zip(bars, [acc_revisiting, acc_fresh]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom",
            color="#1F2A4F", fontweight="bold")
ax.set_ylim(0, 1)
ax.set_ylabel("Final accuracy")
ax.set_title("Scenario B â€” revisiting vs. fresh class accuracy")
plt.tight_layout()
plt.show()

print(f"Revisiting classes:  {sorted(revisiting_B)}")
print(f"Acc (revisiting):    {acc_revisiting:.4f}")
print(f"Acc (fresh):         {acc_fresh:.4f}")
print(f"Gap (fresh - rev):   {acc_fresh - acc_revisiting:+.4f}")

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>7.3 |</span> Overlap matrix visualization</b></p></div>


In [ ]:
def build_overlap_matrix(benchmark, n_cls=100):
    n_tasks = benchmark.nb_experiences
    mat     = np.zeros((n_cls, n_tasks), dtype=int)
    for t in range(n_tasks):
        for c in benchmark.train_stream[t].classes_in_this_experience:
            mat[c, t] = 1
    return mat

mat_A = build_overlap_matrix(bench_A)
mat_B = build_overlap_matrix(bench_B)
mat_C = build_overlap_matrix(bench_C)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ["A â€” disjoint", "B â€” partial overlap", "C â€” exact replay"]
for ax, mat, title in zip(axes, [mat_A, mat_B, mat_C], titles):
    sns.heatmap(mat, ax=ax, cmap=["#FFFFFF", "#4A5FBF"],
                linewidths=0, cbar=False,
                xticklabels=range(mat.shape[1]), yticklabels=False)
    ax.set_title(title)
    ax.set_xlabel("Task")
    if ax is axes[0]:
        ax.set_ylabel("Class")
plt.suptitle("Class--task overlap matrices", fontweight="bold", color="#1F2A4F")
plt.tight_layout()
plt.show()

# <b><span style='color:#1F2A4F'>8 | DISCUSSION</span></b>


<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>8.1 |</span> Where simple replay struggles</b></p></div>

- Replay samples past examples uniformly — it has no concept of "this class is revisiting".
- When a class re-appears, the model sees additional training signal, but the buffer does not increase its weight.
- The revisiting signal is diluted by all other past classes in the buffer.
- On Scenario C (exact replay at end-of-stream), the buffer naturally over-represents the revisiting classes after the final task — explaining any accuracy boost there.
- On Scenario B (partial overlap spread across tasks), the revisiting signal arrives too diffusely for the buffer to adapt.
- Simple replay cannot distinguish beneficial repetition from coincidental overlap.

> <span style='color:#4A5FBF'><b>Key takeaway —</b></span> Replay is overlap-agnostic: it neither exploits revisitation nor is significantly harmed by it.

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>8.2 |</span> Implications for trunk-and-branch methods</b></p></div>

- Trunk-and-branch architectures maintain a shared feature extractor (trunk) and per-task heads (branches).
- When a class revisits, the trunk can be updated with consolidated signal — the branch for that class can be merged or reinforced.
- This is precisely the design space Hemati et al. (2023) motivate: class-incremental learning *with repetition* rewards methods that track class identity across tasks.
- The gap in §7.2 (revisiting vs. fresh accuracy) is the natural target metric for trunk-and-branch methods.
- Methods that ignore class identity across tasks will exhibit the same gap as replay.
- Methods that exploit it should narrow or reverse the gap — revisiting classes should *improve* relative to fresh ones.

<div style="color:white;display:fill;border-radius:8px;background-color:#4A5FBF;font-size:130%;letter-spacing:0.8px"><p style="padding:8px;color:white;"><b><span style='color:#7B8EE8'>8.3 |</span> Extensibility — what to try next</b></p></div>

- Replace ResNet-18 with a ViT-tiny backbone; overlap effects may differ with attention-based representations.
- Vary `min_gap` in `RevisitSpec` to study how temporal distance between revisits affects forgetting.
- Implement a class-aware buffer that oversamples revisiting classes relative to their buffer slot count.
- Plug in LwF (Learning without Forgetting) as an alternative baseline — it also does not exploit overlap.
- Use `exp.overlap_with_previous` to quantify how much each task's classes overlap with prior tasks.
- The PILOT codebase (`sun-hailong/LAMDA-PILOT`) provides many baselines that plug into the `train_one_scenario` loop with minimal changes.

# <b><span style='color:#1F2A4F'>9 | CONCLUSION</span></b>

- CLOVER makes it straightforward to build reproducible class-overlap benchmarks on CIFAR-100 via both a legacy `OverlapDataManager` API and a modern declarative `StreamSpec` + `RevisitSpec` API.
- Simple replay performs consistently across disjoint and overlapping scenarios, but does not exploit revisitation — the accuracy gap between revisiting and fresh classes remains unexploited.
- The next step is to evaluate a trunk-and-branch method (GRAFT) on the same three scenarios, using the revisiting-vs-fresh metric from §7.2 as the primary evaluation criterion.

# <b><span style='color:#1F2A4F'>10 | REFERENCES</span></b>

- Hemati, H., Borth, D., & van de Weijer, J. (2023). Class-Incremental Learning with Repetition. *Conference on Lifelong Learning Agents (CoLLAs)*.
- Wang, Z., Zhang, Z., Lee, C.-Y., Zhang, H., Sun, R., Ren, X., Su, G., Perot, V., Dy, J., & Pfister, T. (2022). Learning to Prompt for Continual Learning. *CVPR 2022*.
- Sun, H. et al. PILOT — A Pre-trained Model-Based Continual Learning Toolbox. [github.com/sun-hailong/LAMDA-PILOT](https://github.com/sun-hailong/LAMDA-PILOT)
- CLOVER — Class-Overlapping Benchmark Library. [github.com/danushkumar-v/clover-cl](https://github.com/danushkumar-v/clover-cl)